# Setting up
Setting up file paths, folders etc.

In [38]:
import json
from rich.console import Console
from pathlib import Path
import pandas as pd
import numpy as np

con = Console()

# Paths/Folders
root = Path.cwd().parent
raw_data            = root /    "data"     /   "raw"
processed_data      = root /    "data"     /   "processed"
models              = root /    "models"
plots               = root /    "plots"
results             = root /    "results"

# Files and their paths
raw_dataset         =   raw_data        /   "inputs.csv"
cleaned_dataset     =   processed_data  /   "full_processed_dataset.csv"
test_dataset        =   processed_data  /   "test_dataset.csv"
train_dataset       =   processed_data  /   "train_dataset.csv"
total_training_dataset_scaled_path = processed_data / "total_training_dataset_scaled.csv"


# Processing the Data

In [39]:
def clean_csv(raw_dataset_path: Path, 
              cleaned_dataset_path: Path,
              test_dataset_path:Path,
              train_dataset_path:Path,) -> pd.DataFrame:
    """Cleans the raw CSV by removing empty strings and dropping NaNs."""
    
    if not raw_dataset_path.exists():
        raise FileNotFoundError(f"Dataset not found at: {raw_dataset}")
    
    # ---------------------------------------------------------------------
    # replacing design points that have empty values with np.nan and removing them
    df = pd.read_csv(raw_dataset_path)
    df = df.replace(r'^\s*$', np.nan, regex=True).dropna(axis=0, how='any')
    # ---------------------------------------------------------------------
    
    
    # ---------------------------------------------------------------------
    # Physics specific checks
    # dropping the columns that has larger than 5mm in max_deformation
    if "max_deformation" in df.columns:
        before = len(df)
        df = df[df["max_deformation"] <= 5].reset_index(drop=True)
        print(f"[x]    Dropped {before - len(df)} rows with max_deformation > 5")
    # ---------------------------------------------------------------------
    
    
    df.to_csv(cleaned_dataset_path, index=False, encoding='utf-8')
    print(f"[x]    Cleaned dataset saved to {cleaned_dataset_path}")
    
    # seperating from the cleaned df dataframe the needed columns
    total_training_dataset = df.iloc[:,[1, 2, 3, 4, 6, 7, 8, 9, 10, 11]].copy()
    
    # Now we shall measure the range, save the data
    df_min = total_training_dataset.min()
    df_max = total_training_dataset.max()
    
    bounds = {
        "min":df_min.to_dict(),
        "max":df_max.to_dict(),
        "columns":total_training_dataset.columns.to_list()
    }
    with open(processed_data/"scaling_bounds.json",'w') as file:
        json.dump(bounds, file, indent=4)
    print(f"[x]    Scaling bounds saved to {processed_data}/scaling_bounds.json")
    
    # scale to [-1, 1]
    # Formula: 2*(x-min)/(max-min) - 1
    df_scaled = 2 * (total_training_dataset - df_min) / (df_max - df_min) - 1
    df_scaled.to_csv(total_training_dataset_scaled_path, index=False, encoding='utf-8')
    print(f"[x]    Scaled between [-1,1] data saved and also the scaling data saved: {total_training_dataset_scaled_path}")
    
    # Splitting (80/20)
    np.random.seed(42)
    mask = np.random.rand(len(df_scaled)) < 0.8
    
    train_pool = df_scaled[mask].reset_index(drop=True)
    test_set  = df_scaled[~mask].reset_index(drop=True)
    
    test_set.to_csv(test_dataset_path, index=False, encoding='utf-8')
    print(f"[x]    Test saved to {test_dataset_path}")
    
    # Generating Nested training sets from the scaled training pools
    shuffled_train = train_pool.sample(frac=1, random_state=42).reset_index(drop=True)
    
    sizes = [300, 600, 900, 1200, 1500]
    for size in sizes:
        subset = shuffled_train.iloc[:size,:]
        filename = f"train_{size}.csv"
        subset.to_csv(processed_data / filename, index=False, encoding='utf-8')
        print(f"[x]    Nested scaled train set saved: {filename} ({len(subset)} samples)")
                

    return df

In [40]:

    
if __name__=="__main__":
    clean_csv(
        raw_dataset_path = raw_dataset,
        cleaned_dataset_path= cleaned_dataset,
        test_dataset_path=test_dataset,
        train_dataset_path=train_dataset
    )

[x]    Dropped 77 rows with max_deformation > 5
[x]    Cleaned dataset saved to c:\Users\Administrator\Desktop\code\comp_design_fw\data\processed\full_processed_dataset.csv
[x]    Scaling bounds saved to c:\Users\Administrator\Desktop\code\comp_design_fw\data\processed/scaling_bounds.json
[x]    Scaled between [-1,1] data saved and also the scaling data saved: c:\Users\Administrator\Desktop\code\comp_design_fw\data\processed\total_training_dataset_scaled.csv
[x]    Test saved to c:\Users\Administrator\Desktop\code\comp_design_fw\data\processed\test_dataset.csv
[x]    Nested scaled train set saved: train_300.csv (300 samples)
[x]    Nested scaled train set saved: train_600.csv (600 samples)
[x]    Nested scaled train set saved: train_900.csv (900 samples)
[x]    Nested scaled train set saved: train_1200.csv (1200 samples)
[x]    Nested scaled train set saved: train_1500.csv (1450 samples)
